In [ ]:
import numpy as np
import netket as nk
import jax
import matplotlib.pyplot as plt
import json
import flax.serialization as serialization
from scipy.ndimage import uniform_filter1d
import os
from tqdm import tqdm
from scipy.optimize import brentq

In [ ]:
N = 25
N_A = N
hamiltonian_idx = 0

if N==N_A:
    data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
else: 
    data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
with open(data_file, "r") as f:
    data = json.load(f)
hamiltonian_file = f"../data/hamiltonians.json"
with open(hamiltonian_file, "r") as f:
    data_h = json.load(f)

T_array      = np.array(data["T"])
energy_results      = data["energy"]
entropy_results     = data["entropy"]
free_energy_results = data["free_energy"]
param_index  = data["param_index"]
J_ZZ = data_h['hamiltonian'][hamiltonian_idx]['J_ZZ']
J_XX = data_h['hamiltonian'][hamiltonian_idx]['J_XX']
h_x = data_h['hamiltonian'][hamiltonian_idx]['h_x']
h_z = data_h['hamiltonian'][hamiltonian_idx]['h_z']

print(f"J_ZZ = {J_ZZ}, J_XX = {J_XX}, h_x = {h_x}, h_z = {h_z}")

In [ ]:
from netket.operator.spin import sigmax, sigmaz

def hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended):
    H_system = 0
    H_extended = 0
    for i in range(N):
        H_system+= h_x * sigmax(hi_system, i)
        H_system+= h_z * sigmaz(hi_system, i)
        H_system += J_ZZ * sigmaz(hi_system, i) @ sigmaz(hi_system, (i + 1) % N)
        H_system += J_XX * sigmax(hi_system, i) @ sigmax(hi_system, (i + 1) % N)
        H_extended += h_x * sigmax(hi_extended, i)
        H_extended += h_z * sigmaz(hi_extended, i)
        H_extended += J_ZZ * sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i + 1) % N)
        H_extended += J_XX * sigmax(hi_extended, i) @ sigmax(hi_extended, (i + 1) % N)
    return H_system, H_extended

hi_system = nk.hilbert.Spin(s=1/2, N=N)
hi_extended = nk.hilbert.Spin(s=1/2, N=2*N)
H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended)

In [ ]:
save = False
jump_lines = False
derivative = False 
fontsize = 14

if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")

def set_ylim_robust(ax, data_arrays, percentile=97):
    all_vals = np.concatenate([np.array(d).flatten() for d in data_arrays])
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin = np.percentile(all_vals, 100 - percentile)
    vmax = np.percentile(all_vals, percentile)
    margin = 0.1 * (vmax - vmin)
    ax.set_ylim(vmin - margin, vmax + margin)

def smooth_and_diff(y, x, window=20):
    y_smooth = uniform_filter1d(np.array(y, dtype=float), size=window, mode='nearest')
    return np.gradient(y_smooth, x)

H_matrix = H_system.to_dense()
eigvals, eigvecs = np.linalg.eigh(H_matrix)

def compute_jump_temperatures(evals, tol=1e-10):
    unique_evals = []
    for e in evals:
        if not unique_evals or abs(e - unique_evals[-1]) > tol:
            unique_evals.append(e)
    unique_evals = np.array(unique_evals)
    T_jumps = []
    for n in range(1, len(unique_evals)):
        weights = unique_evals[n] - unique_evals[:n]
        E_bar = np.dot(unique_evals[:n], weights) / weights.sum()
        T_jumps.append((unique_evals[n] - E_bar) / 2)
    return np.array(T_jumps)

def canonical_ensemble(eigvals, eigvecs, T):
    log_pops = -eigvals / T
    log_pops -= np.max(log_pops)
    pops = np.exp(log_pops)
    pops /= pops.sum()
    energy = np.sum(pops * eigvals)
    mask = pops > 1e-15
    S_VN = -np.sum(pops[mask] * np.log(pops[mask]))
    return energy - T * S_VN, energy, S_VN

def renyi_exact(T, eigvals, tol=1e-12):
    from scipy.optimize import brentq

    eigvals = np.array(eigvals, dtype=float)

    def mre_weights(E_perp):
        w = np.maximum(0.0, E_perp - eigvals)
        Z = w.sum()
        if Z < tol:
            return None
        return w / Z

    def f_root(E_perp):
        w = mre_weights(E_perp)
        if w is None:
            return E_perp
        E_bar = np.dot(w, eigvals)
        return E_perp - (2 * T + E_bar)

    E_min = eigvals.min()
    E_max = eigvals.max() + 4 * T

    try:
        E_perp = brentq(f_root, E_min, E_max, xtol=1e-12)
    except ValueError:
        w = np.zeros(len(eigvals))
        w[0] = 1.0
    else:
        w = mre_weights(E_perp)

    E = np.dot(w, eigvals)
    purity = np.sum(w**2)
    S2 = -np.log(purity)
    F = E - T * S2

    return F, E, S2

temperatures = np.linspace(0.05, 4, 200)
F_arr, E_arr, S_arr = np.array([canonical_ensemble(eigvals, eigvecs, T) for T in temperatures]).T
F_R, E_R, S2_R = np.array([renyi_exact(T, eigvals) for T in temperatures]).T

if jump_lines:
    T_jumps = compute_jump_temperatures(eigvals)
    T_jumps_visible = T_jumps[T_jumps <= temperatures[-1]]
cmap = plt.cm.tab10

def add_jump_lines(ax):
    for i, Tn in enumerate(T_jumps_visible):
        ax.axvline(Tn, color=cmap(i % 10), lw=1.0, ls='--', alpha=0.7)

if derivative:
    SMOOTH_WINDOW = 20
    PERCENTILE    = 97

    dF_dT_renyi = smooth_and_diff(free_energy_results, T_array, window=SMOOTH_WINDOW)
    dE_dT_renyi = smooth_and_diff(energy_results,      T_array, window=SMOOTH_WINDOW)
    dS_dT_renyi = smooth_and_diff(entropy_results,     T_array, window=SMOOTH_WINDOW)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

axes[0,0].plot(temperatures, F_arr, color='C0', label=r'$F_\mathrm{Gibbs}$')
axes[0,0].plot(temperatures, F_R, color='green', label=r'$F_\mathrm{Renyi} \, (ED)$')
axes[0,0].plot(T_array, free_energy_results, 'x', color='C1', markersize=5, alpha=0.9,
               label=r'$F_\mathrm{Renyi}$')

axes[0,0].set_ylabel(r'$F$')
axes[0,0].set_title(r'(a) Free energy')
axes[0,0].set_xlabel(r'$T$')
axes[0,0].legend(fontsize=fontsize-1)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(temperatures, E_arr, color='C0', label=r'$\langle H\rangle_\mathrm{Gibbs}$')
axes[0,1].plot(temperatures, E_R, color='green', label=r'$\langle H\rangle_\mathrm{Renyi} \, (ED)$')
axes[0,1].plot(T_array, energy_results, 'x', color='C1', markersize=5, alpha=0.9,
               label=r'$\langle H\rangle_\mathrm{Renyi}$')

axes[0,1].set_ylabel(r'$\langle H\rangle$')
axes[0,1].set_title(r'(b) Energy')
axes[0,1].set_xlabel(r'$T$')
axes[0,1].legend(fontsize=fontsize-1)
axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(temperatures, S_arr, color='C0', label=r'$S_\mathrm{VN}$')
axes[0,2].plot(temperatures, S2_R, color='green', label=r'$S_\mathrm{Renyi} \, (ED)$')
axes[0,2].plot(T_array, entropy_results, 'x', color='C1', markersize=5, alpha=0.9,
               label=r'$S_2$')

axes[0,2].set_ylabel(r'$S$')
axes[0,2].set_title(r'(c) Entropy')
axes[0,2].set_xlabel(r'$T$')
axes[0,2].legend(fontsize=fontsize-1)
axes[0,2].grid(True, alpha=0.3)

if derivative:
    axes[1,0].scatter(T_array, dF_dT_renyi, color='C1', s=8, zorder=3,
                      label=r'$dF/dT\ \mathrm{(Renyi)}$')

    set_ylim_robust(axes[1,0], [dF_dT_renyi], percentile=PERCENTILE)
    axes[1,0].set_xlabel(r'$T$')
    axes[1,0].set_ylabel(r'$dF/dT$')
    axes[1,0].set_title(r'$dF/dT$')
    axes[1,0].legend(fontsize=fontsize-1)
    axes[1,0].grid(True, alpha=0.3)

    axes[1,1].scatter(T_array, dE_dT_renyi, color='C1', s=8, zorder=3,
                      label=r'$dE/dT\ \mathrm{(Renyi)}$')

    set_ylim_robust(axes[1,1], [dE_dT_renyi], percentile=PERCENTILE)
    axes[1,1].set_xlabel(r'$T$')
    axes[1,1].set_ylabel(r'$dE/dT$')
    axes[1,1].set_title(r'$d\langle H\rangle/dT$')
    axes[1,1].legend(fontsize=fontsize-1)
    axes[1,1].grid(True, alpha=0.3)

    axes[1,2].scatter(T_array, dS_dT_renyi, color='C1', s=8, zorder=3,
                      label=r'$dS_2/dT\ \mathrm{(Renyi)}$')

    set_ylim_robust(axes[1,2], [dS_dT_renyi], percentile=PERCENTILE)
    axes[1,2].set_xlabel(r'$T$')
    axes[1,2].set_ylabel(r'$dS_2/dT$')
    axes[1,2].set_title(r'$dS_2/dT$')
    axes[1,2].legend(fontsize=fontsize-1)
    axes[1,2].grid(True, alpha=0.3)
else:
    for j in range(3):
        axes[1,j].set_visible(False)

if jump_lines:
    for i in range(2):
        for j in range(3):
            if derivative or (not derivative and i == 0):
                if axes[i,j].get_visible():
                    add_jump_lines(axes[i,j])

fig.tight_layout()
if save:
    save_fig(fig, f"F_E_S_N{N}")
else: 
    plt.show()

In [ ]:
def check_temperatures(N, N_A, hamiltonian_idx):
    if N==N_A:
        data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
    else:
        data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
    with open(data_file, "r") as f:
        data = json.load(f)
    T_list = data["T"]
    return np.array(T_list)

def load_params(N, N_A, hamiltonian_idx, T, vstate):
    if N==N_A:
        data_file = f"../data/{hamiltonian_idx}/N{N}/results_N{N}_vs_T.json"
    else:
        data_file = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/results_N{N}_vs_T.json"
    with open(data_file, "r") as f:
        data = json.load(f)
    T_list = data["T"]
    # buscar T más cercana
    idx = int(np.argmin(np.abs(np.array(T_list) - T)))
    file_idx = data["param_index"][idx]
    if N==N_A:
        filename = f"../data/{hamiltonian_idx}/N{N}/params/params_{file_idx:04d}.msgpack"
    else:
        filename = f"../data/{hamiltonian_idx}/N{N}_NA_{N_A}/params/params_{file_idx:04d}.msgpack"
    with open(filename, "rb") as f:
        params = serialization.from_bytes(vstate.parameters, f.read())
    vstate.parameters = params

def canonical_expectation(O, T, eigvals, eigvecs):
    """
    Expectation value <O> en el ensemble canónico a temperatura T.

    Parámetros
    ----------
    O : NetKet operator
    T : float
    eigvals : ndarray
    eigvecs : ndarray

    Returns
    -------
    float
    """

    # convertir operador a matriz densa
    O_matrix = O.to_dense()

    # poblaciones de Boltzmann (estables numéricamente)
    log_pops = -eigvals / T
    log_pops -= np.max(log_pops)

    pops = np.exp(log_pops)
    pops /= pops.sum()

    # <n|O|n>
    O_diag = np.einsum(
        "ij,ji->i",
        np.conj(eigvecs.T),
        O_matrix @ eigvecs,
    )

    # promedio térmico
    return np.sum(pops * O_diag).real

def renyi_expectation(O, vstate):
    """Expectation value <O> en el ensemble de Rényi."""
    return float(vstate.expect(O).mean.real)

def renyi_exact_weights(T, eigvals, tol=1e-12):
    """
    Devuelve los pesos exactos w_k del MRE (alpha=2).
    """

    from scipy.optimize import brentq

    eigvals = np.array(eigvals, dtype=float)

    def mre_weights(E_perp):
        w = np.maximum(0.0, E_perp - eigvals)
        Z = w.sum()
        if Z < tol:
            return None
        return w / Z

    def f_root(E_perp):
        w = mre_weights(E_perp)
        if w is None:
            return E_perp
        E_bar = np.dot(w, eigvals)
        return E_perp - (2 * T + E_bar)

    E_min = eigvals.min()
    E_max = eigvals.max() + 4 * T

    try:
        E_perp = brentq(f_root, E_min, E_max, xtol=1e-12)
        w = mre_weights(E_perp)

    except ValueError:
        # ground state
        w = np.zeros(len(eigvals))
        w[0] = 1.0

    return w

def renyi_exact_expectation(O, T, eigvals, eigvecs):
    """
    Expectation value <O> en el MRE exacto (alpha=2).
    """

    w = renyi_exact_weights(T, eigvals)

    # operador en base de energía
    O_matrix = O.to_dense()

    O_diag = np.einsum(
        "ij,ji->i",
        np.conj(eigvecs.T),
        O_matrix @ eigvecs,
    ).real

    return float(np.dot(w, O_diag))

In [ ]:
save = False             
jump_temperatures = False    
derivative = False  
fontsize = 14

if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        import os
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")

O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
O_xx = sum(sigmax(hi_system, i) @ sigmax(hi_system, (i+1) % N) 
           for i in range(N)) / N
O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i+1) % N) 
           for i in range(N)) / N

O_x_ext = sum(sigmax(hi_extended, i) for i in range(N)) / N
O_xx_ext = sum(sigmax(hi_extended, i) @ sigmax(hi_extended, (i+1) % N)
               for i in range(N)) / N
O_zz_ext = sum(sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i+1) % N)
               for i in range(N)) / N

sampler = nk.sampler.ARDirectSampler(hi_extended)
model = nk.models.ARNNDense(hilbert=hi_extended, layers=1, features=16, activation=jax.nn.gelu)
vstate = nk.vqs.MCState(sampler, model, n_samples=40960)

T_array_compare = check_temperatures(N, N_A, hamiltonian_idx)

renyi_sx, renyi_sxx, renyi_szz = [], [], []
gibbs_sx, gibbs_sxx, gibbs_szz = [], [], []
exact_renyi_sx, exact_renyi_sxx, exact_renyi_szz = [], [], []

T_filtradas = [T for T in T_array_compare if T >= 0.05]
for T in tqdm(T_filtradas, desc=f"Procesando temperaturas vstate"):
    load_params(N, N_A, hamiltonian_idx, T, vstate)
    renyi_sx.append(renyi_expectation(O_x_ext, vstate))
    renyi_sxx.append(renyi_expectation(O_xx_ext, vstate))
    renyi_szz.append(renyi_expectation(O_zz_ext, vstate))

for T in tqdm(temperatures, desc=f"Procesando temperaturas ED"):
    gibbs_sx.append(canonical_expectation(O_x, T, eigvals, eigvecs))
    gibbs_sxx.append(canonical_expectation(O_xx, T, eigvals, eigvecs))
    gibbs_szz.append(canonical_expectation(O_zz, T, eigvals, eigvecs))
    exact_renyi_sx.append(renyi_exact_expectation(O_x, T, eigvals, eigvecs))
    exact_renyi_sxx.append(renyi_exact_expectation(O_xx, T, eigvals, eigvecs))
    exact_renyi_szz.append(renyi_exact_expectation(O_zz, T, eigvals, eigvecs))

T_plot = T_array_compare[T_array_compare >= 0.05]

if derivative:
    from scipy.signal import savgol_filter
    
    def smooth_and_derivative(y, x, window=21, polyorder=3):
        if len(y) < window:
            window = len(y) if len(y) % 2 == 1 else len(y) - 1
        if window < polyorder + 2:
            window = polyorder + 2 if (polyorder + 2) % 2 == 1 else polyorder + 3
        y_smooth = savgol_filter(y, window, polyorder)
        dy_dx = np.gradient(y_smooth, x)
        return dy_dx
    
    window_size = min(21, len(T_plot) if len(T_plot) % 2 == 1 else len(T_plot) - 1)
    d_renyi_sx_dT = smooth_and_derivative(renyi_sx, T_plot, window=window_size, polyorder=3)
    d_renyi_sxx_dT = smooth_and_derivative(renyi_sxx, T_plot, window=window_size, polyorder=3)
    d_renyi_szz_dT = smooth_and_derivative(renyi_szz, T_plot, window=window_size, polyorder=3)
    
    d_gibbs_sx_dT = smooth_and_derivative(gibbs_sx, temperatures, window=21, polyorder=3)
    d_gibbs_sxx_dT = smooth_and_derivative(gibbs_sxx, temperatures, window=21, polyorder=3)
    d_gibbs_szz_dT = smooth_and_derivative(gibbs_szz, temperatures, window=21, polyorder=3)
    
    d_exact_renyi_sx_dT = smooth_and_derivative(exact_renyi_sx, temperatures, window=21, polyorder=3)
    d_exact_renyi_sxx_dT = smooth_and_derivative(exact_renyi_sxx, temperatures, window=21, polyorder=3)
    d_exact_renyi_szz_dT = smooth_and_derivative(exact_renyi_szz, temperatures, window=21, polyorder=3)

def add_jump_line_temperatures(ax, temperatures_array):
    if jump_temperatures:
        try:
            if 'eigvals' in globals():
                T_jumps = compute_jump_temperatures(eigvals)
                T_jumps_visible = T_jumps[T_jumps <= temperatures_array[-1]]
                cmap = plt.cm.tab10
                for i, Tn in enumerate(T_jumps_visible):
                    ax.axvline(Tn, color=cmap(i % 10), lw=1.0, ls='--', alpha=0.5)
        except:
            pass

if derivative:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax_val_0 = axes[0, 0] if derivative else axes[0]
ax_val_0.plot(temperatures, gibbs_sx, label=r'$\langle\sigma^x\rangle$ Gibbs', color='C0')
ax_val_0.scatter(T_plot, renyi_sx, s=10, color='orange', label=r'$\langle\sigma^x\rangle$ Rényi (vstate)', zorder=3)
ax_val_0.plot(temperatures, exact_renyi_sx, color='green', ls='--', label=r'$\langle\sigma^x\rangle$ Rényi ED')
ax_val_0.set_xlabel(r'$T$')
ax_val_0.set_ylabel(r'$\langle\sigma^x\rangle$')
ax_val_0.set_title(r'(a) $\langle\sigma^x\rangle$')
ax_val_0.legend(fontsize=fontsize-1)
ax_val_0.grid(True, alpha=0.3)
add_jump_line_temperatures(ax_val_0, temperatures)

if derivative:
    ax_der_0 = axes[1, 0]
    ax_der_0.scatter(T_plot, d_renyi_sx_dT, s=15, color='orange', 
                     label=r'$d\langle\sigma^x\rangle/dT$ Rényi (vstate)', zorder=3)
    ax_der_0.plot(temperatures, d_gibbs_sx_dT, color='C0', 
                  label=r'$d\langle\sigma^x\rangle/dT$ Gibbs')
    ax_der_0.plot(temperatures, d_exact_renyi_sx_dT, color='green', ls='--', 
                  label=r'$d\langle\sigma^x\rangle/dT$ Rényi ED')
    ax_der_0.set_xlabel(r'$T$')
    ax_der_0.set_ylabel(r'$d\langle\sigma^x\rangle/dT$')
    ax_der_0.set_title(r'Derivada $\langle\sigma^x\rangle$')
    ax_der_0.legend(fontsize=fontsize-1)
    ax_der_0.grid(True, alpha=0.3)
    add_jump_line_temperatures(ax_der_0, temperatures)

ax_val_1 = axes[0, 1] if derivative else axes[1]
ax_val_1.plot(temperatures, gibbs_sxx, label=r'$\Gamma^{x,x}$ Gibbs', color='C0')
ax_val_1.scatter(T_plot, renyi_sxx, s=10, color='orange', label=r'$\Gamma^{x,x}$ Rényi (vstate)', zorder=3)
ax_val_1.plot(temperatures, exact_renyi_sxx, color='green', ls='--', label=r'$\Gamma^{x,x}$ Rényi ED')
ax_val_1.set_xlabel(r'$T$')
ax_val_1.set_ylabel(r'$\Gamma^{x,x}$')
ax_val_1.set_title(r'(b) $\Gamma^{x,x}$')
ax_val_1.legend(fontsize=fontsize-1)
ax_val_1.grid(True, alpha=0.3)
add_jump_line_temperatures(ax_val_1, temperatures)

if derivative:
    ax_der_1 = axes[1, 1]
    ax_der_1.scatter(T_plot, d_renyi_sxx_dT, s=15, color='orange', 
                     label=r'$d\Gamma^{x,x}/dT$ Rényi (vstate)', zorder=3)
    ax_der_1.plot(temperatures, d_gibbs_sxx_dT, color='C0', 
                  label=r'$d\Gamma^{x,x}/dT$ Gibbs')
    ax_der_1.plot(temperatures, d_exact_renyi_sxx_dT, color='green', ls='--', 
                  label=r'$d\Gamma^{x,x}/dT$ Rényi ED')
    ax_der_1.set_xlabel(r'$T$')
    ax_der_1.set_ylabel(r'$d\Gamma^{x,x}/dT$')
    ax_der_1.set_title(r'Derivada $\Gamma^{x,x}$')
    ax_der_1.legend(fontsize=fontsize-1)
    ax_der_1.grid(True, alpha=0.3)
    add_jump_line_temperatures(ax_der_1, temperatures)

ax_val_2 = axes[0, 2] if derivative else axes[2]
ax_val_2.plot(temperatures, gibbs_szz, label=r'$\Gamma^{z,z}$ Gibbs', color='C0')
ax_val_2.scatter(T_plot, renyi_szz, s=10, color='orange', label=r'$\Gamma^{z,z}$ Rényi (vstate)', zorder=3)
ax_val_2.plot(temperatures, exact_renyi_szz, color='green', ls='--', label=r'$\Gamma^{z,z}$ Rényi ED')
ax_val_2.set_xlabel(r'$T$')
ax_val_2.set_ylabel(r'$\Gamma^{z,z}$')
ax_val_2.set_title(r'(c) $\Gamma^{z,z}$')
ax_val_2.legend(fontsize=fontsize-1)
ax_val_2.grid(True, alpha=0.3)
add_jump_line_temperatures(ax_val_2, temperatures)

if derivative:
    ax_der_2 = axes[1, 2]
    ax_der_2.scatter(T_plot, d_renyi_szz_dT, s=15, color='orange', 
                     label=r'$d\Gamma^{z,z}/dT$ Rényi (vstate)', zorder=3)
    ax_der_2.plot(temperatures, d_gibbs_szz_dT, color='C0', 
                  label=r'$d\Gamma^{z,z}/dT$ Gibbs')
    ax_der_2.plot(temperatures, d_exact_renyi_szz_dT, color='green', ls='--', 
                  label=r'$d\Gamma^{z,z}/dT$ Rényi ED')
    ax_der_2.set_xlabel(r'$T$')
    ax_der_2.set_ylabel(r'$d\Gamma^{z,z}/dT$')
    ax_der_2.set_title(r'Derivada $\Gamma^{z,z}$')
    ax_der_2.legend(fontsize=fontsize-1)
    ax_der_2.grid(True, alpha=0.3)
    add_jump_line_temperatures(ax_der_2, temperatures)

plt.tight_layout()

if save:
    save_fig(fig, f"Observables_N{N}")
else:
    plt.show()

In [ ]:
save = True
jump_temperatures = False
derivative = False
fontsize = 14

N_list = [4, 6, 8, 10, 12]
temperatures = np.linspace(0.05, 4.0, 200)

def gibbs_thermodynamic_limit(T_array, J, h):
    """
    Calcula observables en el límite termodinámico para el modelo de Ising
    con campo transversal usando la solución analítica.
    """
    Nk = 4000
    k = np.linspace(0, np.pi, Nk)
    dk = np.pi / Nk
    energies_td = []
    sx_td = []
    szz_td = [] 
    for T in T_array:
        beta = 1.0 / T
        eps_k = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
        tanh_term = np.tanh(beta * eps_k / 2)
        # Energía por sitio 
        e = - np.sum((eps_k / 2) * tanh_term) * dk / np.pi
        # <σx>
        mx = np.sum((h - J * np.cos(k)) / eps_k * tanh_term) * dk / np.pi
        # <σz σz>
        corr = np.sum((J - h * np.cos(k))/eps_k* tanh_term) * dk / np.pi
        energies_td.append(e)
        szz_td.append(corr*2)
        sx_td.append(mx*2)   
    return np.array(energies_td), np.array(sx_td), np.array(szz_td)

print("Calculando límite termodinámico de Gibbs...")
energies_td, sx_td, szz_td = gibbs_thermodynamic_limit(temperatures, -J_ZZ, -h_x)
results = {}

for N in N_list:
    print(f"\nCalculando MRE N = {N}...")
    hi_system = nk.hilbert.Spin(s=1/2, N=N)
    hi_extended = nk.hilbert.Spin(s=1/2, N=N)  
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, 0, h_x, 0, N, hi_system, hi_extended)
    
    O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
    O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i+1) % N) 
               for i in range(N)) / N
    
    H_matrix = H_system.to_dense()
    eigvals, eigvecs = np.linalg.eigh(H_matrix)
    
    energies_renyi = []
    sx_renyi = []
    szz_renyi = []
    energies_gibbs = []
    sx_gibbs = []
    szz_gibbs = []
    energy_density = []
    
    for T in tqdm(temperatures, desc=f"Temperaturas N={N}"):
        F, E_R, S2_R = renyi_exact(T, eigvals) 
        energy_density.append(E_R / N)
        
        sx_renyi.append(renyi_exact_expectation(O_x, T, eigvals, eigvecs))
        szz_renyi.append(renyi_exact_expectation(O_zz, T, eigvals, eigvecs))
        
        sx_gibbs.append(canonical_expectation(O_x, T, eigvals, eigvecs))
        szz_gibbs.append(canonical_expectation(O_zz, T, eigvals, eigvecs))
    
    results[N] = {
        'energy_density': np.array(energy_density),
        'sx_renyi': np.array(sx_renyi),
        'szz_renyi': np.array(szz_renyi),
    }

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(N_list)))

ax1 = axes[0]
for idx, N in enumerate(N_list):
    color = colors[idx]
    ax1.plot(results[N]['energy_density'], results[N]['sx_renyi'], 
             '-', color=color, linewidth=2, label=f'N={N} (MRE)')
ax1.plot(energies_td, sx_td, 'k-', linewidth=2.5, label=r'Gibbs ($N\to\infty$)')
ax1.set_xlabel(r'$\langle H \rangle / N$')
ax1.set_ylabel(r'$\langle \sigma^x \rangle$')
ax1.set_title(r'(a) $\langle\sigma^x\rangle$')
ax1.legend(fontsize=fontsize-1)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
for idx, N in enumerate(N_list):
    color = colors[idx]
    ax2.plot(results[N]['energy_density'], results[N]['szz_renyi'], 
             '-', color=color, linewidth=2, label=f'N={N} (MRE)')
ax2.plot(energies_td, szz_td, 'k-', linewidth=2.5, label=r'Gibbs ($N\to\infty$)')
ax2.set_xlabel(r'$\langle H \rangle / N$')
ax2.set_ylabel(r'$\Gamma^{z,z}$')
ax2.set_title(r'(b) $\Gamma^{z,z}$')
ax2.legend(fontsize=fontsize-1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()

if save:
    import matplotlib
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })
    
    def save_fig(fig, name, subdir="plots/MRE"):
        import os
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")
    
    save_fig(fig, "convergence_MRE_to_Gibbs")
else:
    plt.show()

In [ ]:
N_list = [3, 4, 6, 10]
N_A_list = N_list

fig, axes = plt.subplots(1, 3, figsize=(15, 4))  

exact_drawn = True

for N, N_A in zip(N_list, N_A_list):
    T_array = check_temperatures(N, N_A, hamiltonian_idx)
    T_array = T_array[T_array >= 0.05]
    if N == 10:
        exact_drawn = False
    print(f"\nProcesando N = {N}")

    hi_system = nk.hilbert.Spin(s=1/2, N=N)
    hi_extended = nk.hilbert.Spin(s=1/2, N=2*N)
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended)

    # diagonalización exacta
    if not exact_drawn:
        H_matrix = H_system.to_dense()
        eigvals, eigvecs = np.linalg.eigh(H_matrix)

    O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
    O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i + 1) % N) for i in range(N)) / N
    O_xx = sum(sigmax(hi_system, i) @ sigmax(hi_system, (i + 1) % N) for i in range(N)) / N  
    O_x_ext = sum(sigmax(hi_extended, i) for i in range(N)) / N
    O_zz_ext = sum(sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i + 1) % N) for i in range(N)) / N
    O_xx_ext = sum(sigmax(hi_extended, i) @ sigmax(hi_extended, (i + 1) % N) for i in range(N)) / N 

    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=1, features=16, activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    if N>40:
        vstate.chunk_size = 256*4


    energies_renyi = []
    sx_renyi = []
    szz_renyi = []
    sxx_renyi = []  
    energies_exact = []
    sx_exact = []
    szz_exact = []
    sxx_exact = []  

    for T in tqdm(T_array, desc=f"N={N}"):
        # Rényi
        load_params(N, N_A, hamiltonian_idx, T, vstate)
        E = vstate.expect(H_extended).mean
        e_density = E / N
        mx = vstate.expect(O_x_ext).mean
        corr_zz = vstate.expect(O_zz_ext).mean
        corr_xx = vstate.expect(O_xx_ext).mean 
        energies_renyi.append(e_density)
        sx_renyi.append(mx)
        szz_renyi.append(corr_zz)
        sxx_renyi.append(corr_xx)  

        if not exact_drawn:
            beta = 1.0 / T
            weights = np.exp(-beta * eigvals)
            Z = np.sum(weights)
            E_exact = np.sum(eigvals * weights) / Z
            e_density_exact = E_exact / N
            mx_exact = canonical_expectation(O_x, T, eigvals, eigvecs)
            corr_zz_exact = canonical_expectation(O_zz, T, eigvals, eigvecs)
            corr_xx_exact = canonical_expectation(O_xx, T, eigvals, eigvecs)  
            energies_exact.append(e_density_exact)
            sx_exact.append(mx_exact)
            szz_exact.append(corr_zz_exact)
            sxx_exact.append(corr_xx_exact)  

    
    axes[0].scatter(energies_renyi, sx_renyi, s=15, label=f"N={N}")
    axes[1].scatter(energies_renyi, szz_renyi, s=15, label=f"N={N}")
    axes[2].scatter(energies_renyi, sxx_renyi, s=15, label=f"N={N}")  

    
    if not exact_drawn:
        axes[0].plot(energies_exact, sx_exact, color="gray", linewidth=2, label="exact")
        axes[1].plot(energies_exact, szz_exact, color="gray", linewidth=2, label="exact")
        axes[2].plot(energies_exact, sxx_exact, color="gray", linewidth=2, label="exact") 
        exact_drawn = True


axes[0].set_xlabel(r"$\langle H \rangle / N$")
axes[1].set_xlabel(r"$\langle H \rangle / N$")
axes[2].set_xlabel(r"$\langle H \rangle / N$")

axes[0].set_ylabel(r"$\langle \sigma^x \rangle$")
axes[1].set_ylabel(r"$\langle \sigma^z \sigma^z \rangle$")
axes[2].set_ylabel(r"$\langle \sigma^x \sigma^x \rangle$") 

for ax in axes:
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def compare_p_renyi_gibbs_same_energy(N, N_A, T_dos, hi_extended, H_extended, 
                                      plot=False, xlims=None, ylim=None, n_bins=500):
    # ── 1. Extraer muestras del vstate ──────────────────────────────────────
    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=2, features=24, 
                                activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    
    load_params(N, N_A, hamiltonian_idx, T_dos, vstate)
    vstate.n_samples = 40960
    vstate.chunk_size = 2**8

    E_loc = np.real(vstate.local_estimators(H_extended).reshape(-1))
    E_loc_per_site = E_loc / N
    expect_H = vstate.expect(H_extended).mean.real / N

    # Histograma del vstate
    counts, bin_edges = np.histogram(E_loc_per_site, bins=n_bins, density=True)
    E_vstate = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    p_vstate = counts
    dE_vstate = E_vstate[1] - E_vstate[0]
    p_vstate /= p_vstate.sum() * dE_vstate

    def compute_variance(E_arr, p, dE):
        mean = np.sum(E_arr * p) * dE
        variance = np.sum((E_arr - mean)**2 * p) * dE
        return mean, variance

    mean_vstate, var_vstate = compute_variance(E_vstate, p_vstate, dE_vstate)

    # ── 2. Densidad de estados exacta (JW, saddle point) ───────────────────
    # def jw_dos_saddle(N, J, h, n_points=2000):
    #     m = np.arange(N)
    #     eps_k = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
    #     E_min = -np.sum(eps_k/2) / N
    #     E_max = np.sum(eps_k/2) / N

    #     def log_Z(beta):
    #         return np.sum(np.log(2*np.cosh(beta * eps_k / 2)))

    #     def mean_E(beta):
    #         return -np.sum(eps_k/2 * np.tanh(beta * eps_k/2)) / N

    #     E_arr = np.linspace(E_min*0.999, E_max*0.999, n_points)
    #     log_g = np.zeros(n_points)
    #     for i, E in enumerate(E_arr):
    #         b_star = brentq(lambda b: mean_E(b) - E, -500, 500, xtol=1e-12)
    #         log_g[i] = b_star * E * N + log_Z(b_star)

    #     dE = E_arr[1] - E_arr[0]
    #     g = np.exp(log_g - log_g.max())
    #     g /= g.sum() * dE
    #     return E_arr, g, dE
    def jw_dos_saddle(N, J, h, n_points=4000):
        m = np.arange(N)
        eps = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
        E_lim = np.sum(eps/2)/N

        log_Z  = lambda b: np.sum(np.log(2*np.cosh(b*eps/2)))
        mean_E = lambda b: -np.sum(eps/2*np.tanh(b*eps/2))/N
        var_H  = lambda b: np.sum((eps/2)**2 / np.cosh(b*eps/2)**2)   # Var(H) total

        E = np.linspace(-E_lim*0.999, E_lim*0.999, n_points)
        lg = np.empty_like(E)
        for i, e in enumerate(E):
            b = brentq(lambda b: mean_E(b) - e, -500, 500, xtol=1e-12)
            lg[i] = log_Z(b) + b*e*N - 0.5*np.log(2*np.pi*var_H(b))   # prefactor gaussiano
        dE = E[1] - E[0]
        g = np.exp(lg - lg.max()); g /= g.sum()*dE
        return E, g, dE
    def exact_dos_tfim(N, J, h, dE=0.005):
        """
        DOS exacta (free-fermion, JW) para TFIM 1D. Válida para h > J.
        Devuelve (E_per_site, g_densidad, dE_site) con la misma convención
        que jw_dos_saddle: E en unidades per-site y sum(g)*dE == 1.
        """
        eps_fn = lambda k: 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(k))
        sectors = [
            (2*np.pi*(np.arange(N)+0.5)/N, 0),   # NS, paridad par
            (2*np.pi*np.arange(N)/N,       1),   # R,  paridad impar (h > J)
        ]

        E_all, g_all = [], []
        for k, par in sectors:
            e = eps_fn(k)
            nb = int(e.sum()/dE) + 3
            g = np.zeros((2, nb)); g[0, 0] = 1.0
            for ej in e:
                s = ej/dE; lo = int(s); f = s - lo
                new = g.copy()
                for p in (0, 1):
                    new[1-p, lo:]   += (1-f)*g[p, :nb-lo]
                    new[1-p, lo+1:] +=  f   *g[p, :nb-lo-1]
                g = new
            E_all.append(-0.5*e.sum() + dE*np.arange(nb))
            g_all.append(g[par])

        # Unir los dos sectores en una rejilla común
        E_flat = np.concatenate(E_all)
        g_flat = np.concatenate(g_all)
        E_grid = np.arange(E_flat.min(), E_flat.max() + 1.5*dE, dE)
        g_grid, _ = np.histogram(
            E_flat,
            bins=np.append(E_grid - dE/2, E_grid[-1] + dE/2),
            weights=g_flat,
        )

        # Conversión a per-site + normalización a densidad
        E_grid_site = E_grid / N
        dE_site     = dE / N
        g_grid      = g_grid / (g_grid.sum() * dE_site)

        return E_grid_site, g_grid, dE_site

    E_jw, g_jw, dE_jw = jw_dos_saddle(N, -J_ZZ, -h_x)

    # ── 3. Distribuciones a energía media fija ─────────────────────────────

    def p_gibbs_from_beta(g, E_arr, beta, N, dE):
        log_w = np.log(g + 1e-300) - beta * E_arr * N
        log_w -= log_w.max()
        w = np.exp(log_w)
        return w / (w.sum() * dE)

    def p_renyi_from_c(g, E_arr, c, dE):
        linear = np.maximum(c - E_arr, 0.0)
        w = g * linear
        return w / (w.sum() * dE)

    # Buscamos beta_G tal que <E>_Gibbs(beta_G) = expect_H
    def find_beta_for_mean(E_target, E_arr, g, N, dE):
        def obj(beta):
            p = p_gibbs_from_beta(g, E_arr, beta, N, dE)
            mean = np.sum(E_arr * p) * dE
            return mean - E_target
        # Si E_target < media de g(E), beta > 0. Si E_target es mayor, beta < 0.
        # Asumimos estado térmico con T>0 (beta > 0).
        beta_max = 10.0
        while obj(beta_max) > 0:
            beta_max *= 2
        return brentq(obj, 1e-10, beta_max)

    # Buscamos c tal que <E>_Renyi(c) = expect_H
    def find_c_for_mean(E_target, E_arr, g, dE):
        def obj(c):
            linear = np.maximum(c - E_arr, 0.0)
            w = g * linear
            Z = w.sum() * dE
            if Z <= 0:
                return -1e9
            mean = np.sum(E_arr * w) * dE / Z
            return mean - E_target
        c_min = E_target + 1e-6
        c_max = E_arr.max() + 10.0
        while obj(c_max) < 0:
            c_max *= 2
        return brentq(obj, c_min, c_max)

    beta_G = find_beta_for_mean(expect_H, E_jw, g_jw, N, dE_jw)
    p_gibbs_fixedE = p_gibbs_from_beta(g_jw, E_jw, beta_G, N, dE_jw)
    mean_gibbs_fixedE, var_gibbs_fixedE = compute_variance(E_jw, p_gibbs_fixedE, dE_jw)

    c_R = find_c_for_mean(expect_H, E_jw, g_jw, dE_jw)
    p_renyi_fixedE = p_renyi_from_c(g_jw, E_jw, c_R, dE_jw)
    mean_renyi_fixedE, var_renyi_fixedE = compute_variance(E_jw, p_renyi_fixedE, dE_jw)

    # Temperaturas efectivas
    T_G = 1.0 / beta_G
    # c_per_site = E_target + 2 T_R / N  =>  T_R = (c - E_target) * N / 2
    T_R = (c_R - expect_H) * N / 2.0

    # ── 4. Plot ────────────────────────────────────────────────────────────
    if plot:
        plt.figure(figsize=(10, 6))
        plt.plot(E_jw, p_gibbs_fixedE, 'b-', lw=2.5,
                label=rf'Gibbs ($T_{{\mathrm{{eff}}}}={T_G:.3f}$)')
        plt.plot(E_jw, p_renyi_fixedE, 'r--', lw=2.5,
                label=rf'Rényi-2 ($T_{{\mathrm{{eff}}}}={T_R:.3f}$)')
        plt.bar(E_vstate, p_vstate, width=dE_vstate, alpha=0.5, color='orange',
                label=rf'vstate ($T_{{\mathrm{{nom}}}}={T_dos:.3f}$)')

        plt.xlabel(r'$E/N$', fontsize=12)
        plt.ylabel(r'$p(E)$', fontsize=12)
        plt.title(rf'$p(E)$ a energía media fija: $\langle E\rangle/N = {expect_H:.4f}$', fontsize=14)
        plt.legend(fontsize=10, loc='best')
        plt.grid(alpha=0.3)

        if xlims is not None:
            plt.xlim(expect_H - xlims[0], expect_H + xlims[1])
        if ylim is not None:
            plt.ylim(0, ylim)

        info_text = (f'$N={N}$\n'
                     f'$\\langle E \\rangle/N = {expect_H:.4f}$\n'
                     f'$T_G = {T_G:.3f}$\n'
                     f'$T_R = {T_R:.3f}$')
        plt.text(0.02, 0.98, info_text, transform=plt.gca().transAxes,
                 fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plt.tight_layout()
        plt.show()

    # ── 5. Resultados ──────────────────────────────────────────────────────
    results = {
        'vstate': {
            'mean': mean_vstate, 'variance': var_vstate, 
            'expect_H': expect_H
        },
        'gibbs_same_energy': {
            'mean': mean_gibbs_fixedE, 'variance': var_gibbs_fixedE, 
            'T_eff': T_G, 'beta': beta_G
        },
        'renyi_same_energy': {
            'mean': mean_renyi_fixedE, 'variance': var_renyi_fixedE, 
            'T_eff': T_R, 'c': c_R
        },
    }
    return results

# Llamada de ejemplo
results = compare_p_renyi_gibbs_same_energy(N, N_A, 2.0, hi_extended, H_extended, 
                                           plot=True)
print("Resultados:")
for k, v in results.items():
    print(f"{k}: mean={v['mean']:.5f}, var={v['variance']:.5e}")
    if 'T_eff' in v:
        print(f"    T_eff = {v['T_eff']:.4f}")

In [ ]:
def compare_p_renyi_gibbs_at_T(N, N_A, T_dos, hi_extended, H_extended,
                               plot=False, xlims=None, ylim=None, n_bins=500, ED=False):
    # ── 1. Muestras del vstate ────────────────────────────────────────────
    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=2, features=24,
                                activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    load_params(N, N_A, hamiltonian_idx, T_dos, vstate)
    vstate.n_samples = 40960 * 4
    vstate.chunk_size = 2**10

    E_loc = np.real(vstate.local_estimators(H_extended).reshape(-1)) / N
    expect_H = vstate.expect(H_extended).mean.real / N

    counts, bin_edges = np.histogram(E_loc, bins=n_bins, density=True)
    E_vstate = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    p_vstate = counts
    dE_vstate = E_vstate[1] - E_vstate[0]
    p_vstate /= p_vstate.sum() * dE_vstate

    def stats(E, p, dE):
        m = np.sum(E * p) * dE
        v = np.sum((E - m)**2 * p) * dE
        return m, v

    mean_vs, var_vs = stats(E_vstate, p_vstate, dE_vstate)

    # ── 2. DOS exacta o JW ────────────────────────────────────────────────
    if ED:
        print("Calculando DOS con Diagonalización Exacta (ED)...")
        # Construimos el Hamiltoniano físico de N spines
        hi_phys = nk.hilbert.Spin(s=1/2, N=N)
        H_phys = 0
        for i in range(N):
            H_phys += h_x * sigmax(hi_phys, i)
            H_phys += J_ZZ * sigmaz(hi_phys, i) @ sigmaz(hi_phys, (i + 1) % N)
        
        # Matriz densa y autovalores
        H_mat = np.asarray(H_phys.to_dense())
        E_vals = np.linalg.eigvalsh(H_mat)
        E_vals_per_site = E_vals / N
        
        # Construimos una rejilla de energías (con un pequeño padding)
        E_min = E_vals_per_site.min()
        E_max = E_vals_per_site.max()
        padding = 0.01 * (E_max - E_min)
        n_points = 3000
        E_jw = np.linspace(E_min - padding, E_max + padding, n_points)
        dE_jw = E_jw[1] - E_jw[0]
        
        # Histograma para obtener g(E)
        counts, _ = np.histogram(E_vals_per_site, bins=n_points, density=True)
        g_jw = counts
        g_jw /= np.sum(g_jw) * dE_jw  # Normalizar
    else:
        print("Calculando DOS con aproximación JW (saddle point)...")
        # def jw_dos(N, J, h, n_points=2000):
        #     m = np.arange(N)
        #     eps_k = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
        #     E_min = -np.sum(eps_k/2) / N
        #     E_max =  np.sum(eps_k/2) / N
        #     log_Z = lambda b: np.sum(np.log(2*np.cosh(b*eps_k/2)))
        #     mean_E = lambda b: -np.sum(eps_k/2*np.tanh(b*eps_k/2)) / N
        #     E = np.linspace(E_min*0.999, E_max*0.999, n_points)
        #     lg = np.array([brentq(lambda b: mean_E(b)-e, -500, 500, xtol=1e-12)*e*N + log_Z(brentq(lambda b: mean_E(b)-e, -500, 500, xtol=1e-12))
        #                    for e in E])
        #     dE = E[1]-E[0]
        #     g = np.exp(lg - lg.max()); g /= g.sum()*dE
        #     return E, g, dE

        def jw_dos_saddle(N, J, h, n_points=4000):
            m = np.arange(N)
            eps = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
            E_lim = np.sum(eps/2)/N

            log_Z  = lambda b: np.sum(np.log(2*np.cosh(b*eps/2)))
            mean_E = lambda b: -np.sum(eps/2*np.tanh(b*eps/2))/N
            var_H  = lambda b: np.sum((eps/2)**2 / np.cosh(b*eps/2)**2)   # Var(H) total

            E = np.linspace(-E_lim*0.999, E_lim*0.999, n_points)
            lg = np.empty_like(E)
            for i, e in enumerate(E):
                b = brentq(lambda b: mean_E(b) - e, -500, 500, xtol=1e-12)
                lg[i] = log_Z(b) + b*e*N - 0.5*np.log(2*np.pi*var_H(b))   # prefactor gaussiano
            dE = E[1] - E[0]
            g = np.exp(lg - lg.max()); g /= g.sum()*dE
            return E, g, dE

        def exact_dos_tfim(N, J, h, dE=0.005):
            """
            DOS exacta (free-fermion, JW) para TFIM 1D. Válida para h > J.
            Devuelve (E_per_site, g_densidad, dE_site) con la misma convención
            que jw_dos_saddle: E en unidades per-site y sum(g)*dE == 1.
            """
            eps_fn = lambda k: 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(k))
            sectors = [
                (2*np.pi*(np.arange(N)+0.5)/N, 0),   # NS, paridad par
                (2*np.pi*np.arange(N)/N,       1),   # R,  paridad impar (h > J)
            ]

            E_all, g_all = [], []
            for k, par in sectors:
                e = eps_fn(k)
                nb = int(e.sum()/dE) + 3
                g = np.zeros((2, nb)); g[0, 0] = 1.0
                for ej in e:
                    s = ej/dE; lo = int(s); f = s - lo
                    new = g.copy()
                    for p in (0, 1):
                        new[1-p, lo:]   += (1-f)*g[p, :nb-lo]
                        new[1-p, lo+1:] +=  f   *g[p, :nb-lo-1]
                    g = new
                E_all.append(-0.5*e.sum() + dE*np.arange(nb))
                g_all.append(g[par])

            # Unir los dos sectores en una rejilla común
            E_flat = np.concatenate(E_all)
            g_flat = np.concatenate(g_all)
            E_grid = np.arange(E_flat.min(), E_flat.max() + 1.5*dE, dE)
            g_grid, _ = np.histogram(
                E_flat,
                bins=np.append(E_grid - dE/2, E_grid[-1] + dE/2),
                weights=g_flat,
            )

            # Conversión a per-site + normalización a densidad
            E_grid_site = E_grid / N
            dE_site     = dE / N
            g_grid      = g_grid / (g_grid.sum() * dE_site)

            return E_grid_site, g_grid, dE_site

        E_jw, g_jw, dE_jw = jw_dos_saddle(N, J_ZZ, h_x)

    # ── 3. Distribuciones a T = T_dos ─────────────────────────────────────
    beta_R = 1.0 / T_dos

    def p_gibbs(g, E, beta, N, dE):
        lw = np.log(g + 1e-300) - beta*E*N
        lw -= lw.max()
        w = np.exp(lw)
        return w / (w.sum()*dE)

    def p_renyi(g, E, beta_R, E_bar, N, dE):
        lin = np.maximum(1.0 - (beta_R/2.0)*N*(E - E_bar), 0.0)
        w = g * lin
        return w / (w.sum()*dE)

    def solve_E_bar(E, g, beta_R, N, dE, tol=1e-10):
        p0 = p_gibbs(g, E, beta_R, N, dE)
        e_bar = np.sum(E*p0)*dE
        for _ in range(1000):
            lin = np.maximum(1.0 - (beta_R/2.0)*N*(E - e_bar), 0.0)
            w = g*lin
            Z = w.sum()*dE
            if Z <= 0: break
            e_new = np.sum(E*w)*dE / Z
            if abs(e_new - e_bar) < tol: break
            e_bar = e_new
        return e_bar

    E_bar_R = solve_E_bar(E_jw, g_jw, beta_R, N, dE_jw)

    p_gibbs_T = p_gibbs(g_jw, E_jw, beta_R, N, dE_jw)
    p_renyi_T = p_renyi(g_jw, E_jw, beta_R, E_bar_R, N, dE_jw)

    mean_G, var_G = stats(E_jw, p_gibbs_T, dE_jw)
    mean_R, var_R = stats(E_jw, p_renyi_T, dE_jw)

    # ── 4. Plot ───────────────────────────────────────────────────────────
    if plot:
        plt.figure(figsize=(10, 6))
        plt.plot(E_jw, p_gibbs_T, 'b-',  lw=2.5,
                 label=rf'Gibbs ($T={T_dos:.3f}$)')
        plt.plot(E_jw, p_renyi_T, 'r--', lw=2.5,
                 label=rf'Rényi-2 ($T={T_dos:.3f}$)')
        plt.bar(E_vstate, p_vstate, width=dE_vstate, alpha=0.5, color='orange',
                label=rf'vstate ($T={T_dos:.3f}$)')
        plt.xlabel(r'$E/N$', fontsize=12)
        plt.ylabel(r'$p(E)$', fontsize=12)
        plt.title(rf'$p(E)$ a $T={T_dos:.3f}$:  $\langle E\rangle_\mathrm{{vstate}}/N={expect_H:.4f}$,  '
                  rf'$\langle E\rangle_R/N={mean_R:.4f}$', fontsize=13)
        plt.legend(fontsize=10); plt.grid(alpha=0.3)
        if xlims is not None:
            plt.xlim(expect_H - xlims[0], expect_H + xlims[1])
        if ylim is not None:
            plt.ylim(0, ylim)
        info = (f'$N={N}$\n'
                rf'$\langle E\rangle_R/N = {mean_R:.4f}$' + '\n' +
                rf'$\langle E\rangle_G/N = {mean_G:.4f}$' + '\n' +
                rf'$\Delta E_R = {expect_H - mean_R:+.4f}$')
        plt.text(0.02, 0.98, info, transform=plt.gca().transAxes,
                 fontsize=10, va='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.tight_layout(); plt.show()

    return {
        'vstate': {'mean': mean_vs, 'var': var_vs, 'expect_H': expect_H},
        'gibbs':  {'mean': mean_G,  'var': var_G},
        'renyi':  {'mean': mean_R,  'var': var_R,  'E_bar': E_bar_R,
                   'delta_E': expect_H - mean_R},
        'T': T_dos,
    }

results = compare_p_renyi_gibbs_at_T(N, N_A, 2.0, hi_extended, H_extended,
                                     plot=True, ED=False, xlims=(0.5,0.5))
print(results)

In [ ]:

from scipy.ndimage import gaussian_filter1d
# ── DOS exacta o JW ───────────────────────────────────────────────────────
def jw_dos_saddle_old(N, J, h, n_points=2000):
    m = np.arange(N)
    eps_k = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(2 * np.pi * (m + 0.5) / N))
    E_min = -np.sum(eps_k / 2) / N
    E_max =  np.sum(eps_k / 2) / N
    log_Z  = lambda b: np.sum(np.log(2 * np.cosh(b * eps_k / 2)))
    mean_E = lambda b: -np.sum(eps_k / 2 * np.tanh(b * eps_k / 2)) / N
    E = np.linspace(E_min * 0.999, E_max * 0.999, n_points)
    lg = np.empty_like(E)
    for i, e in enumerate(E):
        b = brentq(lambda b: mean_E(b) - e, -500, 500, xtol=1e-12)
        lg[i] = b * e * N + log_Z(b)
    dE = E[1] - E[0]
    g = np.exp(lg - lg.max())
    g /= g.sum() * dE
    return E, g, dE

def jw_dos_saddle(N, J, h, n_points=4000):
    m = np.arange(N)
    eps = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
    E_lim = np.sum(eps/2)/N

    log_Z  = lambda b: np.sum(np.log(2*np.cosh(b*eps/2)))
    mean_E = lambda b: -np.sum(eps/2*np.tanh(b*eps/2))/N
    var_H  = lambda b: np.sum((eps/2)**2 / np.cosh(b*eps/2)**2)   # Var(H) total

    E = np.linspace(-E_lim*0.999, E_lim*0.999, n_points)
    lg = np.empty_like(E)
    for i, e in enumerate(E):
        b = brentq(lambda b: mean_E(b) - e, -500, 500, xtol=1e-12)
        lg[i] = log_Z(b) + b*e*N - 0.5*np.log(2*np.pi*var_H(b))   # prefactor gaussiano
    dE = E[1] - E[0]
    g = np.exp(lg - lg.max()); g /= g.sum()*dE
    return E, g, dE

def exact_dos_tfim(N, J, h, dE=0.005):
    """
    DOS exacta (free-fermion, JW) para TFIM 1D. Válida para h > J.
    Devuelve (E_per_site, g_densidad, dE_site) con la misma convención
    que jw_dos_saddle: E en unidades per-site y sum(g)*dE == 1.
    """
    eps_fn = lambda k: 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(k))
    sectors = [
        (2*np.pi*(np.arange(N)+0.5)/N, 0),   # NS, paridad par
        (2*np.pi*np.arange(N)/N,       1),   # R,  paridad impar (h > J)
    ]

    E_all, g_all = [], []
    for k, par in sectors:
        e = eps_fn(k)
        nb = int(e.sum()/dE) + 3
        g = np.zeros((2, nb)); g[0, 0] = 1.0
        for ej in e:
            s = ej/dE; lo = int(s); f = s - lo
            new = g.copy()
            for p in (0, 1):
                new[1-p, lo:]   += (1-f)*g[p, :nb-lo]
                new[1-p, lo+1:] +=  f   *g[p, :nb-lo-1]
            g = new
        E_all.append(-0.5*e.sum() + dE*np.arange(nb))
        g_all.append(g[par])

    # Unir los dos sectores en una rejilla común
    E_flat = np.concatenate(E_all)
    g_flat = np.concatenate(g_all)
    E_grid = np.arange(E_flat.min(), E_flat.max() + 1.5*dE, dE)
    g_grid, _ = np.histogram(
        E_flat,
        bins=np.append(E_grid - dE/2, E_grid[-1] + dE/2),
        weights=g_flat,
    )

    # Conversión a per-site + normalización a densidad
    E_grid_site = E_grid / N
    dE_site     = dE / N
    g_grid      = g_grid / (g_grid.sum() * dE_site)

    return E_grid_site, g_grid, dE_site

def ed_dos(N, J_ZZ, h_x, n_points=2000):
    """Calcula la DOS exacta mediante diagonalización."""
    print(f"Calculando DOS exacta (ED) para N={N}...")
    hi_phys = nk.hilbert.Spin(s=1/2, N=N)
    H_phys = 0
    for i in range(N):
        H_phys += h_x * sigmax(hi_phys, i)
        H_phys += J_ZZ * sigmaz(hi_phys, i) @ sigmaz(hi_phys, (i + 1) % N)
    
    # Diagonalización exacta
    H_mat = np.asarray(H_phys.to_dense())
    E_vals = np.linalg.eigvalsh(H_mat) / N  # Energía por sitio
    
    E_min = E_vals.min()
    E_max = E_vals.max()
    padding = (E_max - E_min) * 0.01 if E_max > E_min else 0.1
    
    E = np.linspace(E_min - padding, E_max + padding, n_points)
    dE = E[1] - E[0]
    
    # Histograma de autovalores
    counts, _ = np.histogram(E_vals, bins=n_points, range=(E[0], E[-1]))
    g = counts.astype(float)
    
    # Suavizado gaussiano (necesario para N pequeños y que brentq no falle)
    g = gaussian_filter1d(g, sigma=3)
    
    # Normalizar
    g /= (g.sum() * dE)
    return E, g, dE


# ── Un diagnóstico por temperatura ────────────────────────────────────────
def diagnose_T(N, N_A, T_dos, hi_extended, H_extended, E_jw, g_jw, dE_jw,
               n_bins=500, n_mc_samples=40960*4):
    """
    Devuelve dict con:
      E_vstate     : <H>/N del vstate entrenado
      T_eff_G      : T de Gibbs a la que la energía media coincide con E_vstate
      T_eff_R      : T de Rényi a la que la energía media coincide con E_vstate
      E_G_at_T     : <H>/N de Gibbs a T = T_dos
      E_R_at_T     : <H>/N de Rényi a T = T_dos
    """
    # 1) Muestras del vstate
    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=2, features=24,
                                activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    load_params(N, N_A, hamiltonian_idx, T_dos, vstate)
    vstate.n_samples = n_mc_samples
    vstate.chunk_size = 2**10

    E_vs = float(vstate.expect(H_extended).mean.real / N)

    # 2) Distribuciones y su media (per-site)
    def p_gibbs(beta):
        lw = np.log(g_jw + 1e-300) - beta * E_jw * N
        lw -= lw.max()
        w = np.exp(lw)
        return w / (w.sum() * dE_jw)

    def p_renyi(c_per_site):
        w = g_jw * np.maximum(c_per_site - E_jw, 0.0)
        return w / (w.sum() * dE_jw)

    def mean_of(p):
        return np.sum(E_jw * p) * dE_jw

    # 3) T_eff^G (inversión a energía fija)
    try:
        E_target_G = np.clip(E_vs, E_jw.min() + 1e-6, E_jw.max() - 1e-6)
        beta_G = brentq(lambda b: mean_of(p_gibbs(b)) - E_target_G, 1e-8, 50.0)
        T_G_eff = 1.0 / beta_G
    except Exception:
        T_G_eff = 0.0 

    # 4) T_eff^R (inversión a energía fija)
    try:
        c_R = brentq(lambda c: mean_of(p_renyi(c)) - E_vs,
                     E_vs + 1e-8, E_jw.max() + 10.0)
        T_R_eff = (c_R - E_vs) * N / 2.0
    except Exception:
        T_R_eff = np.nan

    # 5) Distribuciones a T = T_dos
    beta = 1.0 / T_dos
    E_G_T = mean_of(p_gibbs(beta))

    def solve_Ebar_R():
        e_bar = E_G_T
        for _ in range(1000):
            lin = np.maximum(1.0 - (beta / 2.0) * N * (E_jw - e_bar), 0.0)
            w = g_jw * lin
            Z = w.sum() * dE_jw
            if Z <= 0:
                return np.nan
            e_new = np.sum(E_jw * w) * dE_jw / Z
            if abs(e_new - e_bar) < 1e-10:
                return e_new
            e_bar = e_new
        return e_bar

    Ebar_R = solve_Ebar_R()
    c_R_T = Ebar_R + 2.0 * T_dos / N
    E_R_T = mean_of(p_renyi(c_R_T))

    return {
        'T_nom':       T_dos,
        'E_vstate':    E_vs,
        'T_eff_G':     T_G_eff,
        'T_eff_R':     T_R_eff,
        'E_G_at_T':    E_G_T,
        'E_R_at_T':    E_R_T,
        'dE_G':        E_G_T - E_vs,
        'dE_R':        E_R_T - E_vs,
    }

# ══════════════════════════════════════════════════════════════════════════
# EJECUCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════
ED = False

if ED:
    E_jw, g_jw, dE_jw = ed_dos(N, J_ZZ, h_x, n_points=2000)
else:
    E_jw, g_jw, dE_jw = jw_dos_saddle(N, -J_ZZ, -h_x, n_points=2000)

T_list = np.linspace(0, 4, 41)
diagnostics = {}
for T in T_list:
    print(f"Diagnosing T={T:.3f} ...", end=" ", flush=True)
    try:
        diagnostics[T] = diagnose_T(N, N_A, T, hi_extended, H_extended,
                                    E_jw, g_jw, dE_jw)
        d = diagnostics[T]
        print(f"E_vs={d['E_vstate']:+.4f}  "
              f"T_eff^G={d['T_eff_G']:.3f}  T_eff^R={d['T_eff_R']:.3f}  "
              f"dE_R={d['dE_R']:+.4f}")
    except Exception as e:
        print(f"FAILED: {e}")

# ── Plots ────────────────────────────────────────────────────────────────
Ts     = np.array([T for T in T_list if T in diagnostics])
T_G    = np.array([diagnostics[T]['T_eff_G']  for T in Ts])
T_R    = np.array([diagnostics[T]['T_eff_R']  for T in Ts])
dE_G   = np.array([diagnostics[T]['dE_G']     for T in Ts])
dE_R   = np.array([diagnostics[T]['dE_R']     for T in Ts])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: T_eff vs T_nom
ax = axes[0]
ax.plot(Ts, Ts, 'k--', lw=1, label=r'ideal ($T_{\rm eff}=T_{\rm nom}$)')
ax.plot(Ts, T_G, 'bs-', ms=5, label='Gibbs')
ax.plot(Ts, T_R, 'ro-', ms=5, label='Rényi-2')
ax.set_xlabel(r'$T_{\rm nom}$', fontsize=12)
ax.set_ylabel(r'$T_{\rm eff}$', fontsize=12)
ax.set_title('Temperatura efectiva vs nominal', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Panel 2: |T_eff - T_nom| / T_nom (log)
ax = axes[1]
err_G = np.abs(T_G - Ts) / np.maximum(Ts, 1e-6) # Evitar división por 0 en T=0
err_R = np.abs(T_R - Ts) / np.maximum(Ts, 1e-6)
ax.semilogy(Ts, err_G, 'bs-', ms=5, label='Gibbs')
ax.semilogy(Ts, err_R, 'ro-', ms=5, label='Rényi-2')
ax.axhline(0.10, color='gray', ls=':', lw=1, label=r'10% error')
ax.set_xlabel(r'$T_{\rm nom}$', fontsize=12)
ax.set_ylabel(r'$|T_{\rm eff} - T_{\rm nom}|/T_{\rm nom}$', fontsize=12)
ax.set_title('Error relativo en temperatura', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which='both')

# Panel 3: delta E vs T_nom
ax = axes[2]
ax.axhline(0, color='k', ls='--', lw=1)
ax.plot(Ts, dE_G, 'bs-', ms=5, label=r'$E_G(T) - E_{\rm vstate}$')
ax.plot(Ts, dE_R, 'ro-', ms=5, label=r'$E_{R2}(T) - E_{\rm vstate}$')
ax.set_xlabel(r'$T_{\rm nom}$', fontsize=12)
ax.set_ylabel(r'$\Delta E / N$', fontsize=12)
ax.set_title('Desplazamiento de energía media', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/diagnostico_Teff_vs_T.pdf', bbox_inches='tight')
plt.show()

# ── Print resumen tabular ────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"{'T_nom':>7}  {'E_vstate':>10}  {'T_eff^G':>9}  {'T_eff^R':>9}  "
      f"{'err_G':>7}  {'err_R':>7}  {'dE_R/N':>9}")
print("-" * 80)
for T in Ts:
    d = diagnostics[T]
    eG = abs(d['T_eff_G'] - T) / max(T, 1e-6)
    eR = abs(d['T_eff_R'] - T) / max(T, 1e-6)
    print(f"{T:>7.3f}  {d['E_vstate']:>+10.4f}  "
          f"{d['T_eff_G']:>9.3f}  {d['T_eff_R']:>9.3f}  "
          f"{eG:>6.1%}  {eR:>6.1%}  {d['dE_R']:>+9.4f}")
print("=" * 80)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 1. FÓRMULA EXACTA DE JORDAN-WIGNER (SIN DOS, SIN SADDLE POINT)
# ══════════════════════════════════════════════════════════════════════════
def exact_tfim_thermo(N, J, h, beta):
    """
    Calcula la energía media y la entropía exactas para TFIM 1D.
    Usa log-sum-exp para evitar overflows a beta grandes.
    """
    # Modos k para el sector Anti-Periodico (Neveu-Schwarz) y Periodico (Ramond)
    k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
    eps_ns = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_ns))
    
    k_r = 2 * np.pi * np.arange(N) / N
    eps_r = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_r))
    
    # Logaritmos de las funciones de partición
    log_Z_ns = np.sum(np.log(2 * np.cosh(beta * eps_ns / 2)))
    log_Z_r  = np.sum(np.log(2 * np.cosh(beta * eps_r / 2)))
    
    # Usamos logaddexp para calcular ln(Z_ns + Z_r) sin desbordar
    log_Z_sum = np.logaddexp(log_Z_ns, log_Z_r)
    log_Z_tot = log_Z_sum - np.log(2)
    
    # Probabilidades relativas de cada sector (softmax estable)
    P_ns = np.exp(log_Z_ns - log_Z_sum)
    P_r  = np.exp(log_Z_r - log_Z_sum)
    
    # Energías de cada sector
    E_ns = -np.sum((eps_ns / 2) * np.tanh(beta * eps_ns / 2))
    E_r  = -np.sum((eps_r / 2) * np.tanh(beta * eps_r / 2))
    
    # Energía media total y entropía
    E_tot = P_ns * E_ns + P_r * E_r
    S_tot = log_Z_tot + beta * E_tot
    
    return E_tot / N, S_tot / N


def find_beta_gibbs(E_target, N, J, h):
    """Encuentra la beta (1/T) exacta que da la energía media E_target."""
    def obj(beta):
        E_N, _ = exact_tfim_thermo(N, J, h, beta)
        return E_N - E_target
    
    # Rango de búsqueda: T de 0.01 a 50
    beta_min = 1.0 / 50.0
    beta_max = 1.0 / 0.01
    
    # Comprobamos los límites del rango
    E_min, _ = exact_tfim_thermo(N, J, h, beta_max)
    E_max, _ = exact_tfim_thermo(N, J, h, beta_min)
    
    # Si el VMC da una energía por debajo del ground state exacto (ruido MC)
    if E_target <= E_min:
        return 0.0, np.inf
    if E_target >= E_max:
        return np.nan, np.nan
        
    try:
        beta_opt = brentq(obj, beta_min, beta_max)
        T_opt = 1.0 / beta_opt
        return T_opt, beta_opt
    except ValueError:
        # En caso de fallo numérico extremo cerca del mínimo
        return 0.0, np.inf

# ══════════════════════════════════════════════════════════════════════════
# 2. COMPARACIÓN CON LOS VSTATES ENTRENADOS
# ══════════════════════════════════════════════════════════════════════════
def compare_gibbs_exact_vs_vstates(N, N_A, T_list, hi_extended, H_extended,
                                   h_x, J_ZZ, nn_params):
    E_vmc_list = []
    T_eff_list = []
    
    # Parámetros del Hamiltoniano (ajusta si tu J tiene otro signo)
    J = -J_ZZ  # En la fórmula H = -J sigma_z sigma_z - h sigma_x
    h = -h_x
    
    for T_dos in T_list:
        sampler = nk.sampler.ARDirectSampler(hi_extended)
        model = nk.models.ARNNDense(hilbert=hi_extended, layers=nn_params[0], features=nn_params[1],
                                    activation=jax.nn.gelu)
        vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
        
        load_params(N, N_A, hamiltonian_idx, T_dos, vstate)
        vstate.n_samples = 2**14
        vstate.chunk_size = 2**10
        
        expect_H = float(vstate.expect(H_extended).mean.real / N)
        T_eff, _ = find_beta_gibbs(expect_H, N, J, h)
        
        E_vmc_list.append(expect_H)
        T_eff_list.append(T_eff)

    return E_vmc_list, T_eff_list

# ══════════════════════════════════════════════════════════════════════════
# 3. EJECUCIÓN PARA VARIAS CONFIGURACIONES (N, N_A, T_list)
# ══════════════════════════════════════════════════════════════════════════
N_list = [2, 6, 12, 20, 25, 50]
N_A_list = [2, 6, 12, 20, 25, 50]
T_lists = [np.linspace(0.1, 4.0, 40) for _ in N_list]
nn_params = [[1, 16] for _ in N_list]

T_lists[-2] =  np.linspace(0, 2.9, 30)
nn_params[-2] = [2, 24]

if not (len(N_list) == len(N_A_list) == len(T_lists)):
    raise ValueError("N_list, N_A_list y T_lists deben tener la misma longitud")

comparison_results = []
for N, N_A, T_list, nn_param in zip(N_list, N_A_list, T_lists, nn_params):
    hi_system = nk.hilbert.Spin(s=1/2, N=N)
    hi_extended = nk.hilbert.Spin(s=1/2, N=N + N_A)
    H_system, H_extended = hamiltonian_system_and_extended(
        J_ZZ, J_XX, h_x, h_z, N, hi_system, hi_extended
    )

    E_vmc, T_eff = compare_gibbs_exact_vs_vstates(
        N, N_A, T_list, hi_extended, H_extended, h_x, J_ZZ, nn_param
    )
    comparison_results.append({
        "N": N,
        "N_A": N_A,
        "T_list": np.array(T_list),
        "E_vmc": E_vmc,
        "T_eff": T_eff,
    })

# Una sola figura para todas las configuraciones
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
T_all = np.concatenate([result["T_list"] for result in comparison_results])
T_grid = np.linspace(max(0.05, T_all.min()), T_all.max() * 1.2, 200)

for index, result in enumerate(comparison_results):
    N = result["N"]
    N_A = result["N_A"]
    T_list = result["T_list"]
    label = f"N={N}"
    color = f"C{index % 10}"
    J = -J_ZZ
    h = -h_x
    E_exact_grid = [exact_tfim_thermo(N, J, h, 1 / T)[0] for T in T_grid]

    # axes[0].plot(T_grid, E_exact_grid, "--", color=color,
    #              label=f"Gibbs exacto, {label}")
    axes[0].plot(T_list, result["E_vmc"], "o", color=color,
                 label=f"VMC, {label}")
    axes[1].plot(T_list, result["T_eff"], "o-", color=color,
                 label=label)

axes[1].plot(T_grid, T_grid, "k--", linewidth=1,
             label=r"ideal ($T_{\rm eff}=T_{\rm nom}$)")
axes[0].set_xlabel(r"$T$", fontsize=12)
axes[0].set_ylabel(r"$\langle E \rangle / N$", fontsize=12)
axes[0].set_title("Energía media", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[1].set_xlabel(r"$T_{\rm nom}$", fontsize=12)
axes[1].set_ylabel(r"$T_{\rm eff}$", fontsize=12)
axes[1].set_title("Temperatura efectiva", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# ══════════════════════════════════════════════════════════════════════════
# 1. FÓRMULA EXACTA (Sectores NS y R)
# ══════════════════════════════════════════════════════════════════════════
def exact_tfim_thermo(N, J, h, beta):
    """Energía y entropía exactas incluyendo los sectores NS y R."""
    k_ns = 2 * np.pi * (np.arange(N) + 0.5) / N
    eps_ns = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_ns))
    
    k_r = 2 * np.pi * np.arange(N) / N
    eps_r = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k_r))
    
    log_Z_ns = np.sum(np.log(2 * np.cosh(beta * eps_ns / 2)))
    log_Z_r  = np.sum(np.log(2 * np.cosh(beta * eps_r / 2)))
    
    log_Z_sum = np.logaddexp(log_Z_ns, log_Z_r)
    log_Z_tot = log_Z_sum - np.log(2)
    
    P_ns = np.exp(log_Z_ns - log_Z_sum)
    P_r  = np.exp(log_Z_r - log_Z_sum)
    
    E_ns = -np.sum((eps_ns / 2) * np.tanh(beta * eps_ns / 2))
    E_r  = -np.sum((eps_r / 2) * np.tanh(beta * eps_r / 2))
    
    E_tot = P_ns * E_ns + P_r * E_r
    S_tot = log_Z_tot + beta * E_tot
    
    return E_tot / N, S_tot / N

# ══════════════════════════════════════════════════════════════════════════
# 2. APROXIMACIÓN SADDLE POINT (Solo sector NS continuo)
# ══════════════════════════════════════════════════════════════════════════
def jw_dos_saddle(N, J, h, n_points=4000):
    m = np.arange(N)
    eps = 2*np.sqrt(J**2 + h**2 - 2*J*h*np.cos(2*np.pi*(m+0.5)/N))
    E_lim = np.sum(eps/2)/N

    log_Z  = lambda b: np.sum(np.log(2*np.cosh(b*eps/2)))
    mean_E = lambda b: -np.sum(eps/2*np.tanh(b*eps/2))/N
    var_H  = lambda b: np.sum((eps/2)**2 / np.cosh(b*eps/2)**2)   # Var(H) total

    E = np.linspace(-E_lim*0.999, E_lim*0.999, n_points)
    lg = np.empty_like(E)
    for i, e in enumerate(E):
        b = brentq(lambda b: mean_E(b) - e, -500, 500, xtol=1e-12)
        lg[i] = log_Z(b) + b*e*N - 0.5*np.log(2*np.pi*var_H(b))   # prefactor gaussiano
    dE = E[1] - E[0]
    g = np.exp(lg - lg.max()); g /= g.sum()*dE
    return E, g, dE

def thermo_from_dos(N, E_arr, g, dE, beta):
    """Calcula la energía media integrando la DOS con el peso de Boltzmann."""
    # Usamos log para evitar underflow a beta grandes
    log_w = np.log(g + 1e-300) - beta * E_arr * N
    log_w -= np.max(log_w)
    w = np.exp(log_w)
    
    Z = np.sum(w) * dE
    E_mean = np.sum(E_arr * w) * dE / Z
    return E_mean

# ══════════════════════════════════════════════════════════════════════════
# 3. COMPARACIÓN Y PLOT
# ══════════════════════════════════════════════════════════════════════════
def compare_saddle_vs_exact(N_list, J, h, T_grid):
    fig, axes = plt.subplots(1, len(N_list), figsize=(6*len(N_list), 4.5))
    if len(N_list) == 1: axes = [axes]
    
    for ax, N in zip(axes, N_list):
        print(f"Calculando para N={N}...")
        
        # 1. Calcular curvas exactas
        E_exact = np.array([exact_tfim_thermo(N, J, h, 1/T)[0] for T in T_grid])
        
        # 2. Calcular aproximación saddle point
        E_jw, g_jw, dE_jw = jw_dos_saddle(N, J, h, n_points=2000)
        E_saddle = np.array([thermo_from_dos(N, E_jw, g_jw, dE_jw, 1/T) for T in T_grid])
        
        # 3. Errores
        delta_E = np.abs(E_saddle - E_exact)
        
        # 4. Plot Energía vs T
        ax.plot(T_grid, E_exact, 'k-', lw=3, label='Gibbs Exacto (NS + R)')
        ax.plot(T_grid, E_saddle, 'r--', lw=2, label='Aprox. Saddle Point (NS)')
        
        # 5. Plot Error en el eje secundario
        ax2 = ax.twinx()
        ax2.semilogy(T_grid, delta_E + 1e-16, 'b:', lw=1.5, label='Error |E_saddle - E_exact|')
        
        # Estética
        ax.set_xlabel(r'$T$', fontsize=12)
        ax.set_ylabel(r'$\langle E \rangle / N$', fontsize=12)
        ax.set_title(f'Validación Saddle Point para N={N}', fontsize=13)
        
        # Combinar leyendas
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='best', fontsize=9)
        
        ax.grid(alpha=0.3)
        ax2.set_ylabel('Error absoluto en Energía', color='b', fontsize=10)
        ax2.tick_params(axis='y', labelcolor='b')
    
    plt.tight_layout()
    plt.show()

# ══════════════════════════════════════════════════════════════════════════
# 4. EJECUCIÓN
# ══════════════════════════════════════════════════════════════════════════
# Parámetros del Hamiltoniano (ajusta según tu J_ZZ y h_x)
J = 1.0
h = 1.5
T_grid = np.linspace(0.05, 4.0, 60)

# Comparamos un sistema pequeño (N=12) y uno grande (N=50)
compare_saddle_vs_exact(N_list=[12, 20, 50, 70], J=J, h=h, T_grid=T_grid)

In [ ]:
save = False
fontsize = 14
N_list = [14, 20, 25, 50]
N_A_list = N_list
nn_params_list = [[1, 16] for _ in N_list]

nn_params_list[-2] = [2, 24]
nn_params_list[-1] = [2, 24]


if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        import os
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Límite termodinámico
Nk = 4000
k = np.linspace(0, np.pi, Nk)
dk = np.pi / Nk

J = abs(J_ZZ)
h = abs(h_x)

energies_td = []
sx_td = []         
szz_td = []

T_array = np.linspace(0.01, 4, 30)

for T in T_array:
    beta = 1.0 / T
    eps_k = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
    tanh_term = np.tanh(beta * eps_k / 2)
    # Energía por sitio 
    e = - np.sum((eps_k / 2) * tanh_term) * dk / np.pi
    # <σx>
    mx = np.sum((h - J * np.cos(k)) / eps_k * tanh_term) * dk / np.pi
    # <σz σz>
    corr = np.sum((J - h * np.cos(k))/eps_k* tanh_term) * dk / np.pi
    energies_td.append(e)
    szz_td.append(corr*2)
    sx_td.append(mx*2)   

for N, N_A, nn_params in zip(N_list, N_A_list, nn_params_list):
    T_array = check_temperatures(N, N_A, hamiltonian_idx)
    T_array = T_array[T_array >= 0.05]
    print(f"\nProcesando N = {N}")

    # Hilbert spaces
    hi_system = nk.hilbert.Spin(s=1/2, N=N)
    hi_extended = nk.hilbert.Spin(s=1/2, N=N+N_A)

    # Hamiltonianos
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, 0, h_x, 0, N, hi_system, hi_extended)

    # Observables
    O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
    O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i + 1) % N) for i in range(N)) / N
    O_x_ext = sum(sigmax(hi_extended, i) for i in range(N)) / N
    O_zz_ext = sum(sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i + 1) % N) for i in range(N)) / N

    # Vstate
    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=nn_params[0], features=nn_params[1], activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    if N>40:
        vstate.chunk_size = 256*4

    energies_renyi = []
    sx_renyi = []
    szz_renyi = []

    # Loop en temperatura
    for T in tqdm(T_array, desc=f"N={N}"):
        load_params(N, N_A, hamiltonian_idx, T, vstate)
        if N>20:
            vstate.chunk_size = 256
        E = vstate.expect(H_extended).mean
        e_density = E / N
        mx = vstate.expect(O_x_ext).mean
        corr = vstate.expect(O_zz_ext).mean
        energies_renyi.append(e_density)
        sx_renyi.append(mx)
        szz_renyi.append(corr)

    # Plot Rényi
    axes[0].scatter(energies_renyi, sx_renyi, s=15, label=f"N={N}")
    axes[1].scatter(energies_renyi, szz_renyi, s=15, label=f"N={N}")


axes[0].plot(energies_td, sx_td, color="gray", linewidth=3, label="thermodynamic limit")
axes[1].plot(energies_td, szz_td, color="gray", linewidth=3, label="thermodynamic limit")

axes[0].set_xlabel(r"$\langle H \rangle / N$")
axes[1].set_xlabel(r"$\langle H \rangle / N$")

axes[0].set_ylabel(r"$\langle \sigma^x \rangle$")
axes[1].set_ylabel(r"$\langle \sigma^z \sigma^z \rangle$")

axes[0].grid(True)
axes[1].grid(True)

axes[0].legend()
axes[1].legend()

plt.tight_layout()
if save:
    save_fig(fig, f"Observables_vs_energy_{hamiltonian_idx}")
else:
    plt.show()

In [ ]:
save = False
fontsize = 14
N_list = [50]
N_A_list = [50]

# Límite termodinámico
Nk = 4000
k = np.linspace(0, np.pi, Nk)
dk = np.pi / Nk

J = -(J_ZZ)
h = -(h_x)

energies_td = []
sx_td = []         
szz_td = []

T_array = np.linspace(0.01, 4, 30)

for T in T_array:
    beta = 1.0 / T
    eps_k = 2 * np.sqrt(J**2 + h**2 - 2 * J * h * np.cos(k))
    tanh_term = np.tanh(beta * eps_k / 2)
    # Energía por sitio 
    e = - np.sum((eps_k / 2) * tanh_term) * dk / np.pi
    # <σx>
    mx = np.sum((h - J * np.cos(k)) / eps_k * tanh_term) * dk / np.pi
    # <σz σz>
    corr = np.sum((J - h * np.cos(k))/eps_k* tanh_term) * dk / np.pi
    energies_td.append(e)
    szz_td.append(corr*2)
    sx_td.append(mx*2)   


renyi_results = [] 

for N, N_A in zip(N_list, N_A_list):
    T_array = check_temperatures(N, N_A, hamiltonian_idx)
    T_array = T_array[T_array >= 0.05]
    print(f"\nProcesando N = {N}")

    # Hilbert spaces
    hi_system = nk.hilbert.Spin(s=1/2, N=N)
    hi_extended = nk.hilbert.Spin(s=1/2, N=N+N_A)

    # Hamiltonianos
    H_system, H_extended = hamiltonian_system_and_extended(J_ZZ, 0, h_x, 0, N, hi_system, hi_extended)

    # Observables
    O_x = sum(sigmax(hi_system, i) for i in range(N)) / N
    O_zz = sum(sigmaz(hi_system, i) @ sigmaz(hi_system, (i + 1) % N) for i in range(N)) / N
    O_x_ext = sum(sigmax(hi_extended, i) for i in range(N)) / N
    O_zz_ext = sum(sigmaz(hi_extended, i) @ sigmaz(hi_extended, (i + 1) % N) for i in range(N)) / N

    # Vstate
    sampler = nk.sampler.ARDirectSampler(hi_extended)
    model = nk.models.ARNNDense(hilbert=hi_extended, layers=1, features=16, activation=jax.nn.gelu)
    vstate = nk.vqs.MCState(sampler, model, n_samples=40960)
    if N>40:
        vstate.chunk_size = 256*4

    energies_renyi = []
    sx_renyi = []
    szz_renyi = []

    # Loop en temperatura
    for T in tqdm(T_array, desc=f"N={N}"):
        load_params(N, N_A, hamiltonian_idx, T, vstate)
        if N>50:
            vstate.chunk_size = 256
        E = vstate.expect(H_extended).mean
        e_density = E / N
        mx = vstate.expect(O_x_ext).mean
        corr = vstate.expect(O_zz_ext).mean
        energies_renyi.append(e_density)
        sx_renyi.append(mx)
        szz_renyi.append(corr)
    
    renyi_results.append((N, energies_renyi, sx_renyi, szz_renyi))

In [ ]:
if save:
    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use("pgf")
    matplotlib.rcParams.update({
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "legend.fontsize": fontsize,
        "xtick.labelsize": fontsize,
        "ytick.labelsize": fontsize,
        "figure.titlesize": fontsize,
    })

    def save_fig(fig, name, subdir="plots/MRE"):
        import os
        base = os.path.join("..", subdir)
        os.makedirs(base, exist_ok=True)
        for ext in ("pgf", "pdf"):
            fig.savefig(os.path.join(base, f"{name}.{ext}"), bbox_inches="tight")
        print(f"Guardado: {base}/{name}.[pgf|pdf]")
else:
    import matplotlib.pyplot as plt

discard_list = [1, 20, 0, 1]

# Crear figura
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot resultados de Rényi
for (N, energies_renyi, sx_renyi, szz_renyi), n_discard in zip(renyi_results, discard_list):
    if n_discard > 0:
        energies_trimmed = energies_renyi[:-n_discard] if n_discard > 0 else energies_renyi
        sx_trimmed = sx_renyi[:-n_discard]
        szz_trimmed = szz_renyi[:-n_discard]
    else:
        energies_trimmed = energies_renyi
        sx_trimmed = sx_renyi
        szz_trimmed = szz_renyi
    
    # Dibujar línea conectando los puntos (en orden)
    axes[0].plot(energies_trimmed, sx_trimmed, '-', linewidth=1, alpha=0.5, label=f"N={N}")
    axes[0].scatter(energies_trimmed, sx_trimmed, s=15)
    
    axes[1].plot(energies_trimmed, szz_trimmed, '-', linewidth=1, alpha=0.5, label=f"N={N}")
    axes[1].scatter(energies_trimmed, szz_trimmed, s=15)

# Plot límite termodinámico
axes[0].plot(energies_td, sx_td, color="gray", linewidth=3, label="T.L. Gibbs")
axes[1].plot(energies_td, szz_td, color="gray", linewidth=3, label="T.L. Gibbs")

# Etiquetas y formato
axes[0].set_xlabel(r"$\langle H \rangle / N$")
axes[1].set_xlabel(r"$\langle H \rangle / N$")

axes[0].set_ylabel(r"$\langle \sigma^x \rangle$")
axes[1].set_ylabel(r"$\langle \sigma^z \sigma^z \rangle$")

axes[0].grid(True)
axes[1].grid(True)

axes[0].legend()
axes[1].legend()

plt.tight_layout()

# Guardar o mostrar
if save:
    save_fig(fig, f"Observables_vs_energy_{hamiltonian_idx}")
else:
    plt.show()